## 🌿 Step 1 — LangChain AI Chatbot Setup

This notebook configures your development environment for LangChain + LangSmith + Mistral + Tavily.

**Goals**
- Load API keys securely from `.env`
- Verify connectivity with Mistral LLM
- Prepare for Tavily and LangGraph integration

In [ ]:
# Step 1 — Load Environment Variables
import os
from dotenv import load_dotenv

# Load .env file
load_dotenv()

# Verify environment variables
print("LangSmith:", os.getenv("LANGSMITH_API_KEY"))
print("Tavily:", os.getenv("TAVILY_API_KEY"))
print("Mistral:", os.getenv("MISTRAL_API_KEY"))
print("Tracing:", os.getenv("LANGSMITH_TRACING"))

## 🌕 Step 2 — Test Mistral Connection

This verifies that your `.env` file and **Mistral API key** are working with the  
current LangChain 1.0+ API.  
Note: the keyword is now `model_provider` instead of `provider`.

In [ ]:
# Step 2 — Initialize and Test Mistral (Final)
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
import os

# Initialize the Mistral model with the correct parameter name
model = init_chat_model(
    model_provider="mistralai",   # ✅ new parameter name
    model="open-mistral-7b",      # valid Mistral model
    api_key=os.getenv("MISTRAL_API_KEY")
)

# Send a test message
msg = HumanMessage(content="Give me a fun fact about space exploration.")
response = model.invoke([msg])

# Display model response
print(response.content)


## 🌐 Step 3 — Set Up Tavily Search Tool

We’ll install the community integration, import **TavilySearchResults**,  
and verify that our `TAVILY_API_KEY` from `.env` is working.  
This tool will later be bound to Mistral to let the agent search the web.

In [ ]:
# Step 3 — Install and Import Tavily Tool
#!pip install -qU langchain-community

from langchain_community.tools.tavily_search import TavilySearchResults
import os

# Initialize the search tool (reads your TAVILY_API_KEY from environment)
search = TavilySearchResults()

# Run a quick test query
results = search.run("latest advancements in AI-driven healthcare triage")

print(results[:1])   # Print just the first result to verify connectivity


## 🧰 Step 4 — Assemble Your Tools List

Now that the Tavily search tool works,  
we add it to a `tools` list so LangChain can pass it to agents and LLMs later.  
You should see a confirmation message after this cell.

In [ ]:
# Step 4 — Assemble Your Tools List
tools = [search]

if tools:
    print("✅ Tool list assembled successfully:", [t.name for t in tools])
else:
    print("❌ No tools found – check your Tavily setup.")


## 🌉 Step 5 — Bind Tavily to the Working Mistral Model (Plugin Version)

LangChain ≥ 1.0 moved provider-specific classes into separate packages.  
We’ll import `ChatMistralAI` from `langchain_mistralai` and bind the Tavily tool cleanly.

In [ ]:
# Step 5 — Bind Tavily to the Working Mistral Model (Plugin Version)
from langchain_mistralai import ChatMistralAI
from langchain_core.messages import HumanMessage
import os

# Re-initialize the correct model directly from the plugin
model = ChatMistralAI(
    model="open-mistral-7b",          # ✅ valid, current model name
    api_key=os.getenv("MISTRAL_API_KEY")
)

# Bind the Tavily tool so the model can plan tool calls
model_with_tools = model.bind_tools(tools)

# ---- Test a simple prompt (no tool expected)
resp_simple = model_with_tools.invoke(
    [HumanMessage(content="Say hello in one short sentence.")]
)
print("Simple response:", resp_simple.content)
print("Tool calls (should be empty):", resp_simple.tool_calls)

# ---- Test a prompt that may require Tavily
resp_needs_tool = model_with_tools.invoke(
    [HumanMessage(content="Find recent breakthroughs in AI for emergency medicine.")]
)
print("Needs-tool response content:", repr(resp_needs_tool.content))
print("Planned tool calls:", resp_needs_tool.tool_calls)


## 🧩 Step 6 — Create and Run a ReAct Agent (Updated for LangChain 1.1 / LangGraph 1.0)

We now create and test a **ReAct-style Agent** that combines our  
**Mistral LLM** and the **Tavily Search Tool**.

This cell:
1. Builds the agent using the latest LangChain API  
2. Runs a simple LLM-only query  
3. Runs a search query to verify tool integration  
4. Prints diagnostic info about the returned state

In [ ]:
# Step 6 — Create and Run a ReAct Agent  ✅ (stable for current release)
from langgraph.prebuilt import create_react_agent      # <-- correct import for your venv
from langchain_core.messages import HumanMessage
import pprint

# NOTE: this build expects 'model=' not 'llm='
agent_executor = create_react_agent(model=model, tools=tools)
print("✅ Agent ready:", type(agent_executor))

# --- Simple factual query (no tool needed)
print("\n🟢 Simple Query:")
state_simple = agent_executor.invoke({"messages": [HumanMessage(content="What is the capital of France?")]})
print("Response:", state_simple)

# --- Research-style query (should trigger Tavily)
print("\n🟣 Search Query:")
state_search = agent_executor.invoke(
    {"messages": [HumanMessage(content="List two recent AI breakthroughs relevant to emergency medicine workflows.")]}
)
print("Raw agent state:")
pprint.pprint(state_search)

# --- Optional diagnostics
if isinstance(state_search, dict):
    print("\n🔍 Field breakdown:")
    print("Outputs:", state_search.get("outputs"))
    print("Tool calls:", state_search.get("tool_calls"))
else:
    print("\n⚠️ Non-dict state; printed raw above.")

print("\n✅ Step 6 complete — agent executor verified.")


## 🧠 Step 7 — Stream Agent Responses in Real Time

We’ll now use `.stream()` to watch partial outputs as they’re generated.  
This demonstrates the LangChain ReAct agent’s incremental reasoning loop.

In [ ]:
# Step 7 — Streaming Mode
from langchain_core.messages import HumanMessage

# Prepare a streaming query
messages = [HumanMessage(content="Find three emerging applications of AI in medical imaging and summarize briefly.")]

# Start streaming (auto mode)
stream_iterator = agent_executor.stream(
    {"messages": messages},
    stream_mode="auto"
)

# Iterate and print each streaming step
for step in stream_iterator:
    if "messages" in step:
        msg = step["messages"][0]
        print(msg.content, end="", flush=True)


## 💻 Step 8 — Build a Minimal Streamlit App

For a deployable demo, we wrap the same model + agent into a Streamlit UI.  
Save this block as `ex_w09_d3_langchain.py` and run:

```bash
cd ex_w09_d3_langchain
streamlit run ex_w09_d3_langchain.py
```

In [ ]:
# Step 8 — Streamlit App
# ex_w09_d3_langchain.py
# Full working Streamlit chatbot with guaranteed final message display.

import os
import streamlit as st
from dotenv import load_dotenv
from langchain_mistralai import ChatMistralAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage

# ---------- Setup ----------
load_dotenv()

model = ChatMistralAI(
    model="open-mistral-7b",
    api_key=os.getenv("MISTRAL_API_KEY")
)
search = TavilySearchResults()
tools = [search]
agent_executor = create_react_agent(model=model, tools=tools)

st.set_page_config(page_title="LangChain AI Chatbot", page_icon="🔗", layout="wide")
st.title("🔗 LangChain AI Chatbot")
st.caption("Mistral + Tavily + LangGraph demo")

user_input = st.text_area("💬 Ask a question:", placeholder="e.g. What are recent AI breakthroughs in medicine?")
run_button = st.button("Send")

# ---------- Main ----------
if run_button and user_input.strip():
    st.write("### 🧠 Response:")
    placeholder = st.empty()
    partial = ""
    messages = [HumanMessage(content=user_input)]

    # --- 1️⃣ Stream intermediate steps for visibility ---
    try:
        for step in agent_executor.stream({"messages": messages}, stream_mode="auto"):
            if "tool_calls" in step:
                st.info(f"🔧 Tool call: {step['tool_calls']}")
            elif "actions" in step:
                st.info(f"🧩 Action: {step['actions']}")
            elif "messages" in step:
                msg = step["messages"][0]
                partial = msg.content
                placeholder.markdown(partial + "▌")
    except Exception as e:
        st.error(f"⚠️ Stream error: {e}")

    # --- 2️⃣ Always fetch final result explicitly ---
    try:
        final_state = agent_executor.invoke({"messages": messages})
        # depending on version this may be a dict or object
        if isinstance(final_state, dict):
            final_msg = final_state.get("messages", [])
            if final_msg:
                text = getattr(final_msg[-1], "content", None) or str(final_msg[-1])
            else:
                text = str(final_state)
        else:
            text = getattr(final_state, "content", str(final_state))

        placeholder.markdown(text)
    except Exception as e:
        st.error(f"⚠️ Final invoke error: {e}")

    st.success("✅ Done!")
else:
    st.write("👆 Enter a question and press **Send** to start chatting.")


## 🏁 Step 9 — Summary

✅ Loaded environment variables  
✅ Verified Mistral connection  
✅ Added Tavily search tool  
✅ Created ReAct agent with LangGraph  
✅ Demonstrated streaming and optional Streamlit UI  

You now have a functional **LangChain AI Chatbot** ready for reuse in other projects.